<a href="https://colab.research.google.com/github/carlossanzp4-glitch/SanchezParedes-laboratorios/blob/main/Tarea_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# EJECUTE ESTA CELDA PARA CREAR EL ARCHIVO DE PRUEBA
datos_sucios = """Reporte Confidencial de Ventas
Generado automaticamente por el sistema Legacy
id_transaccion;fecha_venta;cliente;ciudad;monto_usd
1001;2026-04-15; Ana Gomez ;lima;250.50
1002;2026-04-18;carlos ruiz;cusco;Error_Red
1003;2026/05/02; LUIS PEREZ;arequipa;1200.00
1003;2026/05/02; LUIS PEREZ;arequipa;1200.00

1004;2026-05-10;Maria Diaz;LIMA;No_Data
1005;2026-06-20;Jorge Soto;cusco;85.90"""
with open("ventas_legacy.csv", "w") as f:
  f.write(datos_sucios)

Misiones:
1. Utilice la función pd.read_csv() para leer el archivo "ventas_legacy.csv".
2. Configure los parámetros adecuados para que Pandas entienda que el archivo está
separado por punto y coma (;) y que debe ignorar las 2 primeras filas de basura.
3. Guarde el resultado en un DataFrame llamado df_ventas y muéstrelo en pantalla.

In [ ]:
df_ventas = pd.read_csv("ventas_legacy.csv", sep=";", skiprows=2)
df_ventas

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd
0,1001,2026-04-15,Ana Gomez,lima,250.50
1,1002,2026-04-18,carlos ruiz,cusco,Error_Red
2,1003,2026/05/02,LUIS PEREZ,arequipa,1200.00
3,1003,2026/05/02,LUIS PEREZ,arequipa,1200.00
4,1004,2026-05-10,Maria Diaz,LIMA,No_Data
5,1005,2026-06-20,Jorge Soto,cusco,85.90


Parte 2: Exploración y Diagnóstico (Data Profiling)
Antes de limpiar, debemos conocer el estado de salud de nuestro DataFrame.
Misiones:
1. Utilice el método adecuado para imprimir un resumen técnico del DataFrame (cantidad
de columnas, tipos de datos actuales y memoria usada). Pista: Recuerde qué tipo de dato
asume Pandas cuando hay texto mezclado con números.
2. Imprima por pantalla la cantidad exacta de filas y columnas usando el atributo .shape.
3. Utilice el método .isnull().sum() para verificar si Pandas detectó algún valor nulo (NaN) de
forma automática.

In [ ]:
df_ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_transaccion  6 non-null      int64 
 1   fecha_venta     6 non-null      object
 2   cliente         6 non-null      object
 3   ciudad          6 non-null      object
 4   monto_usd       6 non-null      object
dtypes: int64(1), object(4)
memory usage: 372.0+ bytes


In [ ]:
print(df_ventas.shape)

(6, 5)


In [ ]:
df_ventas.isnull().sum()

,0
id_transaccion,0
fecha_venta,0
cliente,0
ciudad,0
monto_usd,0


Parte 3: El Arte del Casteo y Limpieza (Fase TRANSFORM)
Como notó en el paso anterior, Pandas clasificó la fecha y el monto como texto (object).
Debemos transformar el "ADN" de esos datos.
Misiones:
1. Casteo de Fechas: Convierta la columna fecha_venta al tipo Datetime utilizando
pd.to_datetime(). Asegúrese de incluir el parámetro errors='coerce' para proteger el
código.
2. Casteo Numérico: Convierta la columna monto_usd a número utilizando
pd.to_numeric(..., errors='coerce').
3. Manejo de Nulos: Al castear el monto, los textos "Error_Red" y "No_Data" se habrán
convertido en NaN. Utilice el método .fillna() para rellenar esos vacíos con el valor 0.
4. Eliminación de Duplicados: Aplique el método .drop_duplicates(inplace=True) para
eliminar cualquier transacción que se haya registrado dos veces por error.

In [ ]:
df_ventas.drop_duplicates(inplace=True)
df_ventas["fecha_venta"] = df_ventas["fecha_venta"].str.replace("/", "-")

In [ ]:
df_ventas["fecha_venta"] = pd.to_datetime(df_ventas["fecha_venta"], errors="coerce")
df_ventas["monto_usd"] = pd.to_numeric(df_ventas["monto_usd"], errors="coerce")
df_ventas["monto_usd"] = df_ventas["monto_usd"].fillna(0)
df_ventas

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd
0,1001,2026-04-15,Ana Gomez,lima,250.5
1,1002,2026-04-18,carlos ruiz,cusco,0.0
2,1003,2026-05-02,LUIS PEREZ,arequipa,1200.0
4,1004,2026-05-10,Maria Diaz,LIMA,0.0
5,1005,2026-06-20,Jorge Soto,cusco,85.9


Parte 4: Reglas de Negocio y Manipulación de Textos
Ahora que los datos matemáticos están sanos, vamos a estandarizar los textos y crear nuevas

variables.
Misiones:
1. Limpie la columna cliente eliminando los espacios en blanco a los extremos y
convirtiendo todos los nombres a formato Título (ej. "Ana Gomez"). Pista: Utilice
.str.strip() y .str.title().
2. Estandarice la columna ciudad convirtiendo todo el texto a MAYÚSCULAS.
3. Lógica Condicional: Importe la librería numpy. Cree una nueva columna llamada
tipo_cliente utilizando np.where(). La regla es:
○ Si el monto_usd es estrictamente mayor a 500, el cliente es "VIP".
○ De lo contrario, el cliente es "Regular".

In [ ]:
df_ventas["cliente"] = df_ventas['cliente'].str.strip().str.title()
df_ventas

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd
0,1001,2026-04-15,Ana Gomez,lima,250.5
1,1002,2026-04-18,Carlos Ruiz,cusco,0.0
2,1003,2026-05-02,Luis Perez,arequipa,1200.0
4,1004,2026-05-10,Maria Diaz,LIMA,0.0
5,1005,2026-06-20,Jorge Soto,cusco,85.9


In [ ]:
df_ventas['ciudad'] = df_ventas['ciudad'].str.upper()
df_ventas

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd
0,1001,2026-04-15,Ana Gomez,LIMA,250.5
1,1002,2026-04-18,Carlos Ruiz,CUSCO,0.0
2,1003,2026-05-02,Luis Perez,AREQUIPA,1200.0
4,1004,2026-05-10,Maria Diaz,LIMA,0.0
5,1005,2026-06-20,Jorge Soto,CUSCO,85.9


In [ ]:
df_ventas["tipo_cliente"] = np.where(df_ventas['monto_usd'] > 500 , #condicion
                                      "VIP", # en caso es True
                                      "Regular" )# caso es False
df_ventas

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd,tipo_cliente
0,1001,2026-04-15,Ana Gomez,LIMA,250.5,Regular
1,1002,2026-04-18,Carlos Ruiz,CUSCO,0.0,Regular
2,1003,2026-05-02,Luis Perez,AREQUIPA,1200.0,VIP
4,1004,2026-05-10,Maria Diaz,LIMA,0.0,Regular
5,1005,2026-06-20,Jorge Soto,CUSCO,85.9,Regular


Parte 5: Agregación y Reporte Final
Para finalizar el proceso ETL, prepararemos un resumen gerencial.
Misiones:
1. Extraiga el mes de la venta utilizando el accesor de fechas (.dt.month) y guárdelo en una
nueva columna llamada mes_operacion.
2. Utilice el método .groupby() para agrupar las ventas por mes_operacion y sume (.sum())
la columna monto_usd. Guarde este resumen en la variable reporte_mensual.
3. Muestre en pantalla el reporte_mensual resultante utilizando display().

In [ ]:
df_ventas["mes_operacion"] = df_ventas['fecha_venta'].dt.month
df_ventas

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd,tipo_cliente,mes_operacion
0,1001,2026-04-15,Ana Gomez,LIMA,250.5,Regular,4
1,1002,2026-04-18,Carlos Ruiz,CUSCO,0.0,Regular,4
2,1003,2026-05-02,Luis Perez,AREQUIPA,1200.0,VIP,5
4,1004,2026-05-10,Maria Diaz,LIMA,0.0,Regular,5
5,1005,2026-06-20,Jorge Soto,CUSCO,85.9,Regular,6


In [ ]:
reporte_mensual = df_ventas.groupby("mes_operacion")["monto_usd"].sum().reset_index()
display(reporte_mensual)

,mes_operacion,monto_usd
0,4,250.5
1,5,1200.0
2,6,85.9
